In [6]:
import os
import getpass
import requests
import json

# 1. Configuración segura de la API Key
print("Configuración de seguridad:")
api_key = getpass.getpass("Ingresa tu Gemini API Key: ").strip()

# 2. Fraccionamiento del problema: Función central de diagnóstico
def diagnosticar_averia(descripcion_usuario):
    """
    Toma la descripción del usuario y devuelve una ficha técnica.
    Usa Fast Prompting y busca dinámicamente el modelo compatible en tu cuenta.
    """
    
    # FAST PROMPTING: Rol directo, contexto mínimo, formato estricto y restricciones.
    system_prompt = """
    Rol: Director Técnico Virtual de Servii AI.
    Tarea: Traduce el problema del usuario a una Ficha Técnica.
    Formato estricto (no agregues nada más):
    - Diagnóstico:
    - Profesional:
    - Urgencia (Alta/Media/Baja):
    - Materiales:
    - Pregunta clave:
    Restricciones: Cero charla, sin saludos, sé directo.
    """
    
    prompt_completo = f"{system_prompt}\n\nAVERÍA DEL USUARIO:\n{descripcion_usuario}"
    
    try:
        # PASO A: Buscamos dinámicamente el modelo disponible en tu cuenta
        list_url = f"https://generativelanguage.googleapis.com/v1beta/models?key={api_key}"
        list_res = requests.get(list_url)
        
        if list_res.status_code != 200:
            return f"Error al validar API Key: {list_res.text}"
        
        models_data = list_res.json().get("models", [])
        
        # Filtramos los modelos que soporten generación de texto y sean rápidos (flash)
        available_models = [
            m["name"] for m in models_data 
            if "generateContent" in m.get("supportedGenerationMethods", [])
            and "flash" in m["name"] and "2.5" not in m["name"]
        ]
        
        # Si encuentra un modelo, usa ese. Si no, usa el genérico por defecto.
        if available_models:
            modelo_elegido = available_models[0]
        else:
            modelo_elegido = "models/gemini-1.5-flash-latest"
            
        # PASO B: Hacemos la consulta al modelo que encontramos
        gen_url = f"https://generativelanguage.googleapis.com/v1beta/{modelo_elegido}:generateContent?key={api_key}"
        headers = {"Content-Type": "application/json"}
        
        # Estructura optimizada de tokens y temperatura
        payload = {
            "contents": [{"parts": [{"text": prompt_completo}]}],
            "generationConfig": {
                "temperature": 0.2,      
                "maxOutputTokens": 150   
            }
        }
        
        response = requests.post(gen_url, headers=headers, json=payload)
        res_json = response.json()
        
        if response.status_code == 200:
            return res_json['candidates'][0]['content']['parts'][0]['text']
        else:
            return f"Error de la API en {modelo_elegido}: {res_json.get('error', {}).get('message', response.text)}"
            
    except Exception as e:
        return f"Error general al procesar: {e}"

# 3. BONUS ADICIONAL: Interactividad solicitada en la consigna
def iniciar_simulador():
    print("-" * 50)
    print("🏠 BIENVENIDO A SERVII AI - SIMULADOR DE DIAGNÓSTICO (GEMINI) 🏠")
    print("-" * 50)
    
    # Input interactivo
    input_usuario = input("\nDescribe el problema que tienes en tu hogar:\n> ")
    
    print("\n⏳ Buscando modelo compatible y procesando solicitud...")
    
    resultado = diagnosticar_averia(input_usuario)
    
    print("\n✅ FICHA TÉCNICA GENERADA:\n")
    print(resultado)
    print("-" * 50)

# Ejecutamos el simulador
iniciar_simulador()

Configuración de seguridad:
--------------------------------------------------
🏠 BIENVENIDO A SERVII AI - SIMULADOR DE DIAGNÓSTICO (GEMINI) 🏠
--------------------------------------------------

⏳ Buscando modelo compatible y procesando solicitud...

✅ FICHA TÉCNICA GENERADA:

- Diagnóstico: Fuga
--------------------------------------------------
